# ur5e-reach-ddpg-her on a Kaggle GPU

Trains the repo's configs (DDPG + HER, tutorial settings) on Kaggle's GPU, then evaluates (100 test episodes),
records videos and plots. Everything lands in this notebook's output (`/kaggle/working`).

**How to run**
1. Kaggle -> *Create* -> *New Notebook* -> *File* -> *Import Notebook* -> upload this file.
2. Right panel -> *Session options*: **Accelerator: GPU T4 x2** (or P100), **Internet: on**
   (needs a phone-verified Kaggle account).
3. Edit `JOBS` / `STEPS` below if you like, then **Save Version -> Save & Run All (Commit)**. It runs in the
   background; you can close the tab.
4. When it finishes, open the version -> *Output* -> download `results.zip` (or the whole `ur5e-reach-ddpg-her` folder).

**Resuming** (only if a run hits Kaggle's 12 h limit): *Add Input* -> *Your Work* -> pick the previous version's
output, then commit again. The notebook copies its `runs/` (checkpoints every 10k steps) and continues.


In [ ]:
JOBS = ["panda_ddpg:0,1,2", "ur5e_ddpg:0"]   # config:seeds  (configs/*.yaml in the repo)
STEPS = 100_000                               # per run
PARALLEL = 4                                  # jobs at once (Kaggle: 4 CPU cores; 2 jobs share each T4)
REPO = "https://github.com/GugulothBhuvan/ur5e-reach-ddpg-her"
BRANCH = "main"

In [ ]:
import glob, os, shutil, subprocess, time
T0 = time.time()
sh = lambda c: subprocess.run(c, shell=True, check=True)
W = "/kaggle/working/ur5e-reach-ddpg-her"
if not os.path.exists(W):
    sh(f"git clone --depth 1 -b {BRANCH} {REPO} {W}")
else:
    sh(f"cd {W} && git pull --ff-only")
# earlier version attached as input -> resume its runs
for prev in glob.glob("/kaggle/input/**/ur5e-reach-ddpg-her/runs/*_s[0-9]*", recursive=True):
    dst = os.path.join(W, "runs", os.path.basename(prev))
    if not os.path.exists(dst):
        shutil.copytree(prev, dst)
        print("resuming from", prev)
sh("apt-get -qq update && apt-get -qq install -y libosmesa6 > /dev/null")   # headless MuJoCo rendering
sh("pip install -q stable-baselines3==2.9.2 sb3-contrib==2.9.2 gymnasium==1.4.0 panda-gym==3.0.7 "
   "mujoco==3.17.0 imageio imageio-ffmpeg pyyaml")                          # keeps Kaggle's CUDA torch
import torch; print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
hours_left = 11.4 - (time.time() - T0) / 3600 - 0.6   # Kaggle stops at 12 h; leave time for eval + videos
sh(f"cd {W} && python -m pytest -q tests")            # UR5e env sanity checks (~10 s)
sh(f"cd {W} && python run_all.py --jobs {' '.join(JOBS)} --steps {STEPS} --parallel {PARALLEL} --hours {hours_left:.2f}")

In [ ]:
from IPython.display import Image, Markdown, display
s = os.path.join(W, "results", "summary.md")
if os.path.exists(s):
    display(Markdown(open(s).read()))
    display(Image(os.path.join(W, "results", "plots", "learning_curves.png")))
shutil.make_archive("/kaggle/working/results", "zip", W, "results")
print("training logs:", sorted(os.listdir(os.path.join(W, "logs"))))